In [1]:
import numpy as np

from embedder import Embedder
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index, VectorSearch

## Q1. Embedding a query

In [2]:
embedder = Embedder()

query = "How does approximate nearest neighbor search work?"
v = embedder.encode(query)

print("Shape:", v.shape)
print("First value:", v[0])

Shape: (384,)
First value: -0.02058203437252893


## Loading the data

In [3]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

print("Number of documents:", len(documents))
print(documents[0].keys())

Number of documents: 72
dict_keys(['content', 'filename'])


## Q2. Cosine similarity

In [14]:
target_filename = (
    "02-vector-search/lessons/07-sqlitesearch-vector.md"
)

target_document = next(
    document
    for document in documents
    if document["filename"] == target_filename
)

document_vector = embedder.encode(
    target_document["content"]
)

similarity = document_vector.dot(v)

print("Cosine similarity:", similarity)

Cosine similarity: 0.36107026789538205


## Q3. Chunking and search by hand

In [15]:
chunks = chunk_documents(
    documents,
    size=2000,
    step=1000,
)

print("Number of chunks:", len(chunks))
print(chunks[0].keys())

Number of chunks: 295
dict_keys(['start', 'content', 'filename'])


In [16]:
chunk_contents = [
    chunk["content"]
    for chunk in chunks
]

X = embedder.encode_batch(chunk_contents)

print("Embedding matrix shape:", X.shape)

Embedding matrix shape: (295, 384)


In [17]:
scores = X.dot(v)

best_index = scores.argmax()
best_chunk = chunks[best_index]

print("Best score:", scores[best_index])
print("Filename:", best_chunk["filename"])
print("Chunk start:", best_chunk["start"])

Best score: 0.6489016436447387
Filename: 02-vector-search/lessons/07-sqlitesearch-vector.md
Chunk start: 1000


## Q4. Vector search with minsearch

In [18]:
vector_index = VectorSearch(
    keyword_fields=["filename"]
)

vector_index.fit(X, chunks)

In [19]:
query_q4 = (
    "What metric do we use to evaluate a search engine?"
)

query_q4_vector = embedder.encode(query_q4)

q4_results = vector_index.search(
    query_q4_vector,
    num_results=5,
)

for rank, result in enumerate(q4_results, start=1):
    print(rank, result["filename"], result["start"])

1 04-evaluation/lessons/05-search-metrics.md 0
2 04-evaluation/lessons/01-intro.md 0
3 01-agentic-rag/lessons/05-search.md 0
4 04-evaluation/lessons/01-intro.md 2000
5 04-evaluation/lessons/15-next-steps.md 0


## Q5. Text search vs vector search

In [20]:
text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)

text_index.fit(chunks)

In [21]:
query_q5 = "How do I store vectors in PostgreSQL?"

vector_results = vector_index.search(
    embedder.encode(query_q5),
    num_results=5,
)

text_results = text_index.search(
    query_q5,
    num_results=5,
)

In [22]:
print("Vector search results:")

for rank, result in enumerate(vector_results, start=1):
    print(rank, result["filename"], result["start"])

Vector search results:
1 02-vector-search/lessons/08-pgvector.md 0
2 02-vector-search/lessons/08-pgvector.md 3000
3 03-orchestration/lessons/05-rag.md 2000
4 02-vector-search/lessons/08-pgvector.md 1000
5 02-vector-search/lessons/08-pgvector.md 2000


In [23]:
vector_filenames = {
    result["filename"]
    for result in vector_results
}

text_filenames = {
    result["filename"]
    for result in text_results
}

only_in_vector = vector_filenames - text_filenames

print("Files found only by vector search:")
for filename in only_in_vector:
    print(filename)

Files found only by vector search:
02-vector-search/lessons/08-pgvector.md


 ## Q6. Hybrid search

In [24]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])

            scores[key] = (
                scores.get(key, 0)
                + 1 / (k + rank)
            )

            docs[key] = doc

    ranked_keys = sorted(
        scores,
        key=scores.get,
        reverse=True,
    )

    return [
        docs[key]
        for key in ranked_keys[:num_results]
    ]

In [25]:
query_q6 = "How do I give the model access to tools?"

q6_vector_results = vector_index.search(
    embedder.encode(query_q6),
    num_results=5,
)

q6_text_results = text_index.search(
    query_q6,
    num_results=5,
)

In [26]:
hybrid_results = rrf(
    [
        q6_vector_results,
        q6_text_results,
    ]
)

for rank, result in enumerate(hybrid_results, start=1):
    print(rank, result["filename"], result["start"])

1 01-agentic-rag/lessons/13-function-calling.md 4000
2 01-agentic-rag/lessons/01-intro.md 2000
3 01-agentic-rag/lessons/14-agentic-loop.md 0
4 04-evaluation/lessons/02-ground-truth.md 1000
5 01-agentic-rag/lessons/16-other-frameworks.md 0
